# Pipeline evaluation

Three-layer eval for one recording at a time. Re-run from cell 4 onwards after changing `RECORDING_ID` to evaluate a different recording.

**Inputs** (under `DATA_ROOT`):
- `<id>.wav` — input recording (stereo, 16 kHz)
- `debleed/<id>_{L,R}.wav` — oracle per-speaker channels (raw)
- `debleed_enhanced/<id>_{L,R}.wav` — enhanced oracle (precomputed by `scripts/enhance_clarin_debleed.py`)
- `debleed_enhanced_transcripts/<id>_{L,R}.txt` — reference transcripts (Whisper draft → human-corrected; precomputed by `scripts/transcribe_clarin_debleed.py`)
- `after_pipeline/<id>_{s1,s2}.wav` — pipeline outputs
- `after_pipeline_transcripts/<id>.txt` — pipeline transcript (both speakers)

**Three layers**:
1. **Layer 1 — DER** (`pyannote.metrics`): silero VAD timelines on pipeline outputs vs. enhanced oracle channels; permutation handled by Hungarian solver.
2. **Layer 2 — separation**: torchmetrics intrusive metrics (SI-SDR / PESQ-WB / STOI) against the VAD-gated enhanced oracle, plus SQUIM non-intrusive estimates on the mixture and pipeline outputs.
3. **Layer 3 — ASR**: cpWER + tcpWER via MeetEval (CHiME-7/8 toolkit), pipeline transcript vs. corrected GT. Text is normalized (lowercase + no punctuation, diacritics kept).

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "asr":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import gc
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import torch
from IPython.display import Audio, display

from torchmetrics.functional.audio import (
    scale_invariant_signal_distortion_ratio,
    perceptual_evaluation_speech_quality,
    short_time_objective_intelligibility,
)
from torchaudio.pipelines import SQUIM_OBJECTIVE

from asr_pipeline.stages.separation import _vad_mask_silero
from asr_pipeline.eval import (
    compute_der,
    cpwer_meeteval,
    parse_gt_txt,
    parse_transcript_file,
)

DATA_ROOT = Path("/home/user/datasets/clarin_gotowy/gotowy")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
print(f"DATA_ROOT = {DATA_ROOT}")

available_ids = sorted({p.stem for p in DATA_ROOT.glob("*.wav")})
print(f"available recordings: {available_ids}")

## Choose recording + load all source files

In [ ]:
# --- Knobs ---
RECORDING_ID        = "442dd69e"    # any of `available_ids` printed above
VAD_THRESHOLD       = 0.50          # match pipeline's vad_threshold for fair comparison
VAD_SOFT_THRESHOLD  = 0.20
VAD_ATTACK_FRAMES   = 1
VAD_RELEASE_FRAMES  = 1
TCP_COLLAR_S        = 5.0           # tcpWER per-word time tolerance (CHiME-7 default)


# --- Resolve paths + load audio ---
rid = RECORDING_ID


def _resolve_gt(channel: str) -> tuple[Path, str]:
    """Prefer human-corrected GT in `true_transcripts/`; fall back to the
    Whisper draft in `debleed_enhanced_transcripts/`. Returns (path, source).
    """
    true_p = DATA_ROOT / "true_transcripts" / f"{rid}_{channel}.txt"
    if true_p.exists():
        return true_p, "true_gt"
    draft_p = DATA_ROOT / "debleed_enhanced_transcripts" / f"{rid}_{channel}.txt"
    return draft_p, "whisper_draft"


gt_L_path, gt_L_src = _resolve_gt("L")
gt_R_path, gt_R_src = _resolve_gt("R")

paths = {
    "input":         DATA_ROOT / f"{rid}.wav",
    "oracle_L":      DATA_ROOT / "debleed" / f"{rid}_L.wav",
    "oracle_R":      DATA_ROOT / "debleed" / f"{rid}_R.wav",
    "enhanced_L":    DATA_ROOT / "debleed_enhanced" / f"{rid}_L.wav",
    "enhanced_R":    DATA_ROOT / "debleed_enhanced" / f"{rid}_R.wav",
    "gt_L":          gt_L_path,
    "gt_R":          gt_R_path,
    "pipeline_s1":   DATA_ROOT / "after_pipeline" / f"{rid}_s1.wav",
    "pipeline_s2":   DATA_ROOT / "after_pipeline" / f"{rid}_s2.wav",
    "pipeline_txt":  DATA_ROOT / "after_pipeline_transcripts" / f"{rid}.txt",
}
for label, p in paths.items():
    if not p.exists():
        raise FileNotFoundError(f"missing {label}: {p}")

print(f"GT source — L: {gt_L_src!r:15s}  R: {gt_R_src!r}")
if "whisper_draft" in (gt_L_src, gt_R_src):
    print("  (note: at least one channel is the Whisper draft, not human-verified)")

def _load_mono(path):
    audio, sr_ = sf.read(str(path))
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio.astype(np.float32), sr_

input_mix, sr  = _load_mono(paths["input"])
oracle_L, _    = _load_mono(paths["oracle_L"])
oracle_R, _    = _load_mono(paths["oracle_R"])
enhanced_L, _  = _load_mono(paths["enhanced_L"])
enhanced_R, _  = _load_mono(paths["enhanced_R"])
pipeline_s1, _ = _load_mono(paths["pipeline_s1"])
pipeline_s2, _ = _load_mono(paths["pipeline_s2"])

# Trim everything to the shortest stream so downstream alignment is safe.
min_len = min(
    len(input_mix), len(oracle_L), len(oracle_R),
    len(enhanced_L), len(enhanced_R),
    len(pipeline_s1), len(pipeline_s2),
)
input_mix   = input_mix[:min_len]
oracle_L    = oracle_L[:min_len]
oracle_R    = oracle_R[:min_len]
enhanced_L  = enhanced_L[:min_len]
enhanced_R  = enhanced_R[:min_len]
pipeline_s1 = pipeline_s1[:min_len]
pipeline_s2 = pipeline_s2[:min_len]
total_dur_s = min_len / sr

print(f"recording {rid}: {total_dur_s:.1f}s @ {sr} Hz")
print(f"  input mix      : {len(input_mix)} samples")
print(f"  oracle L / R   : {len(oracle_L)} / {len(oracle_R)} samples (raw)")
print(f"  enhanced L / R : {len(enhanced_L)} / {len(enhanced_R)} samples")
print(f"  pipeline s1/s2 : {len(pipeline_s1)} / {len(pipeline_s2)} samples")

# Quick before/after look at the enhancement on one channel.
t = np.arange(min_len) / sr
fig, axes = plt.subplots(2, 1, figsize=(16, 3), sharex=True, sharey=True)
axes[0].plot(t, oracle_L,   lw=0.4, color="C0"); axes[0].set_ylabel("oracle L (raw)")
axes[1].plot(t, enhanced_L, lw=0.4, color="C1"); axes[1].set_ylabel("enhanced L")
axes[1].set_xlabel("time (s)"); plt.tight_layout(); plt.show()

## Speaker activity timelines (reference via diarization, hypothesis via VAD)

**Reference side** — pyannote diarization on each enhanced oracle channel. Each debleed channel contains the main speaker loudly plus residual crosstalk of the other speaker; running silero VAD on the channel would fire on *both* and give a contaminated reference. Diarizing the channel and keeping only the **higher-RMS (= main) speaker's** turns matches what `scripts/transcribe_clarin_debleed.py` does for GT generation, so the reference signal and reference diarization stay consistent with how the GT transcripts were produced.

**Hypothesis side** — silero VAD on the pipeline output streams. Those streams *are* already per-speaker (separation + assembly), so VAD here legitimately gives each speaker's effective timeline.

Outputs of this cell:
- `gated_L`, `gated_R` — enhanced × main-speaker mask. Reference signal for Layer 2 SI-SDR / PESQ / STOI.
- `mask_L_ref`, `mask_R_ref` — 0/1 sample masks for the same.
- `segs_*_ref`, `segs_s{1,2}_hyp` — start/end segment lists for Layer 1 DER.

In [ ]:
import math
from asr_pipeline.config import load_pipeline_config_from_yaml

# Read pyannote config from the pipeline's own YAML so the model id + HF
# token stay in sync if rotated.
_pipe_cfg = load_pipeline_config_from_yaml("asr_pipeline/configs/default.yaml")

# --- Reference: diarization on enhanced oracle channels ---
print("loading pyannote diarization (reference side)...")
from pyannote.audio import Pipeline as _PyannotePipeline
diarizer = _PyannotePipeline.from_pretrained(
    _pipe_cfg.diarization.model_id, token=_pipe_cfg.diarization.hf_token,
).to(device)


def _diarize_main_speaker_channel(audio, sr_, name):
    """Diarize one channel; pick the higher-RMS speaker as the main one."""
    waveform = torch.from_numpy(audio).unsqueeze(0)
    diar = diarizer(
        {"waveform": waveform, "sample_rate": sr_},
        min_speakers=1, max_speakers=2,
    )
    diar = diar.speaker_diarization if hasattr(diar, "speaker_diarization") else diar

    spk_segs = {}
    for turn, _, spk in diar.itertracks(yield_label=True):
        spk_segs.setdefault(spk, []).append((turn.start, turn.end))
    if not spk_segs:
        print(f"  {name}: no speakers detected")
        return np.zeros(len(audio), dtype=np.float32), []

    rms = {}
    for spk, segs in spk_segs.items():
        sq, n = 0.0, 0
        for s, e in segs:
            i, j = max(0, int(s * sr_)), min(len(audio), int(e * sr_))
            sq += float((audio[i:j] ** 2).sum())
            n += j - i
        rms[spk] = math.sqrt(sq / max(n, 1))
    main_spk = max(rms, key=rms.get)
    print(f"  {name}: main={main_spk}  RMS by spk: " +
          ", ".join(f"{s}={rms[s]:.4f}{'*' if s == main_spk else ''}" for s in sorted(rms, key=lambda x: -rms[x])))

    segs = spk_segs[main_spk]
    mask = np.zeros(len(audio), dtype=np.float32)
    for s, e in segs:
        i, j = max(0, int(s * sr_)), min(len(audio), int(e * sr_))
        mask[i:j] = 1.0
    return mask, segs


print("Diarizing enhanced oracle channels...")
mask_L_ref, _segs_L_main = _diarize_main_speaker_channel(enhanced_L, sr, "oracle L")
mask_R_ref, _segs_R_main = _diarize_main_speaker_channel(enhanced_R, sr, "oracle R")

# Gated oracle = enhanced × main-speaker mask  (reference signal for Layer 2)
gated_L = (enhanced_L * mask_L_ref).astype(np.float32)
gated_R = (enhanced_R * mask_R_ref).astype(np.float32)

# Free pyannote before loading silero.
diarizer = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# --- Hypothesis: silero VAD on per-speaker pipeline output streams ---
print("\nloading silero VAD (hypothesis side)...")
vad_model, _ = torch.hub.load("snakers4/silero-vad", "silero_vad", trust_repo=True)
vad_model = vad_model.to(device)


def _vad_hyp(audio):
    mask, probs = _vad_mask_silero(
        audio, vad_model, device, VAD_THRESHOLD,
        soft_threshold=VAD_SOFT_THRESHOLD,
        attack_frames=VAD_ATTACK_FRAMES,
        release_frames=VAD_RELEASE_FRAMES,
    )
    if len(mask) < len(audio):
        mask = np.concatenate([mask, np.ones(len(audio) - len(mask), dtype=np.float32)])
    return mask, probs


mask_s1_hyp, _ = _vad_hyp(pipeline_s1)
mask_s2_hyp, _ = _vad_hyp(pipeline_s2)

vad_model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def _mask_to_segments(mask, sample_rate, min_dur_s=0.05):
    """Sample-resolution binary mask -> [(start_s, end_s), ...]."""
    pad = np.concatenate(([0.0], mask, [0.0]))
    diff = np.diff((pad > 0.5).astype(np.int8))
    starts = np.where(diff ==  1)[0]
    ends   = np.where(diff == -1)[0]
    segs = []
    for s, e in zip(starts, ends):
        if (e - s) / sample_rate >= min_dur_s:
            segs.append((s / sample_rate, e / sample_rate))
    return segs


segs_L_ref  = _mask_to_segments(mask_L_ref,  sr)
segs_R_ref  = _mask_to_segments(mask_R_ref,  sr)
segs_s1_hyp = _mask_to_segments(mask_s1_hyp, sr)
segs_s2_hyp = _mask_to_segments(mask_s2_hyp, sr)

print()
print(f"speech (oracle L main): {sum(e-s for s, e in segs_L_ref):6.1f}s in {len(segs_L_ref):3d} segments")
print(f"speech (oracle R main): {sum(e-s for s, e in segs_R_ref):6.1f}s in {len(segs_R_ref):3d} segments")
print(f"speech (pipeline s1)  : {sum(e-s for s, e in segs_s1_hyp):6.1f}s in {len(segs_s1_hyp):3d} segments")
print(f"speech (pipeline s2)  : {sum(e-s for s, e in segs_s2_hyp):6.1f}s in {len(segs_s2_hyp):3d} segments")


# Visualise the four timelines.
fig, ax = plt.subplots(figsize=(16, 2.2))
for i, (label, segs, color) in enumerate([
    ("oracle L (main, diar)",  segs_L_ref,  "#1f77b4"),
    ("oracle R (main, diar)",  segs_R_ref,  "#ff7f0e"),
    ("pipeline s1 (VAD)",      segs_s1_hyp, "#2ca02c"),
    ("pipeline s2 (VAD)",      segs_s2_hyp, "#d62728"),
]):
    for s, e in segs:
        ax.barh(i, e - s, left=s, height=0.6, color=color, alpha=0.85)
ax.set_yticks(range(4))
ax.set_yticklabels(["oracle L (main)", "oracle R (main)", "pipeline s1", "pipeline s2"])
ax.set_xlim(0, total_dur_s); ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
ax.set_xlabel("time (s)"); ax.set_title("Speaker activity timelines")
plt.tight_layout(); plt.show()

## Speaker permutation

Pick the (pipeline_si ↔ oracle_X) pairing that maximises total SI-SDR between the pipeline output and the gated oracle. Used for Layer 1 (DER) and Layer 2 (SI-SDR); Layer 3 (cpWER) handles permutation independently.

In [ ]:
# Score both speaker arrangements; pick the higher total SI-SDR.
def _sisdr_t(est, tgt):
    return scale_invariant_signal_distortion_ratio(
        torch.from_numpy(est).float(), torch.from_numpy(tgt).float()
    ).item()

sdr_id = _sisdr_t(pipeline_s1, gated_L) + _sisdr_t(pipeline_s2, gated_R)
sdr_sw = _sisdr_t(pipeline_s1, gated_R) + _sisdr_t(pipeline_s2, gated_L)

if sdr_id >= sdr_sw:
    perm = (0, 1)
    spk_for_estimate = {0: "L", 1: "R"}
    stream_for = {"L": "s1", "R": "s2"}
else:
    perm = (1, 0)
    spk_for_estimate = {0: "R", 1: "L"}
    stream_for = {"L": "s2", "R": "s1"}

print(f"pipeline s1 -> oracle {spk_for_estimate[0]}")
print(f"pipeline s2 -> oracle {spk_for_estimate[1]}")
print(f"total SI-SDR: identity={sdr_id:.2f} dB  swap={sdr_sw:.2f} dB  chosen={max(sdr_id, sdr_sw):.2f} dB")

## Layer 1 — Diarization Error Rate

Reference: oracle VAD timelines (one per channel). Hypothesis: pipeline-output VAD timelines (one per stream). `pyannote.metrics` handles the speaker assignment internally — we report DER with miss / false alarm / confusion broken out.

In [ ]:
# Use oracle labels (L, R) as ref speakers and stable pipeline labels (s1, s2) as hyp speakers.
# pyannote does its own optimal assignment; we don't pre-permute.
der_no_collar  = compute_der(
    ref_segments={"L": segs_L_ref, "R": segs_R_ref},
    hyp_segments={"s1": segs_s1_hyp, "s2": segs_s2_hyp},
    total_duration_s=total_dur_s,
    collar=0.0,
)
der_collar_500 = compute_der(
    ref_segments={"L": segs_L_ref, "R": segs_R_ref},
    hyp_segments={"s1": segs_s1_hyp, "s2": segs_s2_hyp},
    total_duration_s=total_dur_s,
    collar=0.5,
)

def _show_der(name, d):
    print(f"{name:<22}  DER={d['der']*100:5.1f}%   "
          f"miss={d['miss']*100:5.1f}%   "
          f"FA={d['false_alarm']*100:5.1f}%   "
          f"conf={d['confusion']*100:5.1f}%")

_show_der("DER (no collar)",    der_no_collar)
_show_der("DER (collar=500ms)", der_collar_500)

## Layer 2 — separation metrics

**Intrusive** (against the VAD-gated enhanced oracle): SI-SDR, PESQ-WB, STOI plus their improvements over the mono-mix baseline (`SI-SDRi`, `PESQi`, `STOIi`). All via `torchmetrics.functional.audio`, same library as `evaluate.py` and `asr/evaluate_asr.py`.

**Non-intrusive** (no reference): SQUIM (`torchaudio.pipelines.SQUIM_OBJECTIVE`) on the mono mix and on each pipeline output stream. SQUIM predicts STOI / PESQ / SI-SDR from the signal alone — useful here because the "reference" gated oracle is itself enhanced+VAD-gated, not truly clean.

In [ ]:
# Pick the pipeline stream that matches each oracle speaker.
estimate_for_L = pipeline_s1 if perm[0] == 0 else pipeline_s2
estimate_for_R = pipeline_s1 if perm[0] == 1 else pipeline_s2


def _intrusive(estimate, target, mix):
    """SI-SDR / PESQ-WB / STOI + improvements over the mono-mix baseline.

    torchmetrics functional signatures (all positional):
      scale_invariant_signal_distortion_ratio(preds, target)
      perceptual_evaluation_speech_quality(preds, target, fs, mode)
      short_time_objective_intelligibility(preds, target, fs)
    """
    e = torch.from_numpy(estimate).float()
    t = torch.from_numpy(target).float()
    m = torch.from_numpy(mix).float()
    si  = scale_invariant_signal_distortion_ratio(e, t).item()
    si0 = scale_invariant_signal_distortion_ratio(m, t).item()
    # PESQ-WB requires 16 kHz; raises on silent / no-speech inputs.
    try:
        pq  = perceptual_evaluation_speech_quality(e, t, sr, "wb").item()
        pq0 = perceptual_evaluation_speech_quality(m, t, sr, "wb").item()
    except Exception as exc:
        print(f"  PESQ skipped ({type(exc).__name__}: {exc})")
        pq = pq0 = float("nan")
    st  = short_time_objective_intelligibility(e, t, sr).item()
    st0 = short_time_objective_intelligibility(m, t, sr).item()
    return {
        "si_sdr": si, "si_sdr_baseline": si0, "si_sdri": si - si0,
        "pesq":   pq, "pesq_baseline":   pq0, "pesqi":  pq - pq0,
        "stoi":   st, "stoi_baseline":   st0, "stoii":  st - st0,
    }


intr_L = _intrusive(estimate_for_L, gated_L, input_mix)
intr_R = _intrusive(estimate_for_R, gated_R, input_mix)

print("Intrusive (vs VAD-gated enhanced oracle):")
for label, r in (("L", intr_L), ("R", intr_R)):
    print(f"  speaker {label}  (pipeline {stream_for[label]}):")
    print(f"    SI-SDR = {r['si_sdr']:6.2f} dB  (mix: {r['si_sdr_baseline']:6.2f} dB)  Δ = {r['si_sdri']:+6.2f} dB")
    print(f"    PESQ   = {r['pesq']:6.2f}     (mix: {r['pesq_baseline']:6.2f})     Δ = {r['pesqi']:+6.2f}")
    print(f"    STOI   = {r['stoi']:6.3f}     (mix: {r['stoi_baseline']:6.3f})     Δ = {r['stoii']:+6.3f}")


# --- Non-intrusive SQUIM (no reference needed) ---
print("\nloading SQUIM_OBJECTIVE...")
squim_model = SQUIM_OBJECTIVE.get_model().eval().to(device)


@torch.no_grad()
def _squim(audio):
    x = torch.from_numpy(audio).float().unsqueeze(0).to(device)
    stoi_pred, pesq_pred, sisdr_pred = squim_model(x)
    return {
        "squim_stoi":   float(stoi_pred.item()),
        "squim_pesq":   float(pesq_pred.item()),
        "squim_si_sdr": float(sisdr_pred.item()),
    }


squim_mix = _squim(input_mix)
squim_s1  = _squim(pipeline_s1)
squim_s2  = _squim(pipeline_s2)

squim_model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nSQUIM non-intrusive (no reference):")
for label, r in (("mixture", squim_mix), ("pipeline s1", squim_s1), ("pipeline s2", squim_s2)):
    print(f"  {label:13s} STOI={r['squim_stoi']:.3f}  PESQ={r['squim_pesq']:.2f}  SI-SDR={r['squim_si_sdr']:6.2f} dB")

## Layer 3 — cpWER + tcpWER

Reference = corrected GT transcripts in `debleed_enhanced_transcripts/<id>_{L,R}.txt` (one file per oracle channel). Hypothesis = pipeline transcript (both speakers in one file).

**cpWER** tries every assignment of hyp speakers to ref speakers and reports the minimum WER. **tcpWER** additionally requires each word to be within `TCP_COLLAR_S` seconds of its reference position — catches cases where the right text was said by the right speaker but at the wrong time, which plain cpWER silently lets through.

Both come from [MeetEval](https://github.com/fgnt/meeteval) (CHiME-7/8 evaluation toolkit). Text is normalized to lowercase + no punctuation before scoring; Polish diacritics are preserved.

In [ ]:
# Reference: one Utterance list per oracle channel (true GT if present,
# else the Whisper draft — see the path-resolver in the choose-recording cell).
ref_utts = {
    "L": parse_gt_txt(paths["gt_L"]),
    "R": parse_gt_txt(paths["gt_R"]),
}

# Hypothesis: pipeline transcript (one file, two speakers).
hyp_utts = parse_transcript_file(paths["pipeline_txt"])

print(f"Reference (L from {gt_L_src!r}, R from {gt_R_src!r}):")
for spk, utts in ref_utts.items():
    n_words = sum(len(u.text.split()) for u in utts)
    sample  = utts[0].text if utts else ""
    print(f"  {spk}: {len(utts):4d} segments, ~{n_words:5d} words; first: {sample[:100]!r}")

print("\nHypothesis (pipeline transcript):")
for spk, utts in hyp_utts.items():
    n_words = sum(len(u.text.split()) for u in utts)
    sample  = utts[0].text if utts else ""
    print(f"  {spk}: {len(utts):4d} segments, ~{n_words:5d} words; first: {sample[:100]!r}")

result = cpwer_meeteval(ref_utts, hyp_utts, session_id=rid, tcp_collar_s=TCP_COLLAR_S)

print()
print(f"cpWER  = {result['cpwer']*100:5.2f}%   ({result['cp_errors']}/{result['cp_length']} words)")
print(f"tcpWER = {result['tcpwer']*100:5.2f}%   ({result['tcp_errors']}/{result['tcp_length']} words, collar={TCP_COLLAR_S}s)")
print(f"cpWER  assignment: {result['cp_assignment']}")
print(f"tcpWER assignment: {result['tcp_assignment']}")

## Layer 3b — ORC-WER baseline (naïve Whisper on the raw mixture)

The cpWER above tells you what the **full pipeline** achieves. To know whether the pipeline is *useful* you need a no-pipeline baseline: take the raw mixture, hand it straight to Whisper as a single stream, and score with **ORC-WER** (Optimal Reference Combination WER — MeetEval).

ORC-WER scores a single-stream hypothesis against a multi-speaker reference by finding the optimal assignment of hyp words to reference speakers, with overlap-tolerance built in (during a 2-speaker overlap region either speaker's hyp word can match either ref speaker's word). It gives the strongest reasonable WER for the mixture baseline — i.e., the bar the pipeline has to beat.

Cached: the mixture transcript is saved to `eval_cache/mixture_transcripts/<id>.json` so re-running the cell is instant.

In [ ]:
import whisper
from meeteval.io.seglst import SegLST
from meeteval.wer import orcwer
from asr_pipeline.eval.metrics import _normalize_text

MIX_CACHE = DATA_ROOT / "eval_cache" / "mixture_transcripts" / f"{rid}.json"
MIX_CACHE.parent.mkdir(parents=True, exist_ok=True)

if MIX_CACHE.exists():
    print(f"loading cached mixture transcript: {MIX_CACHE}")
    with open(MIX_CACHE, "r", encoding="utf-8") as f:
        mix_segments = json.load(f)["segments"]
else:
    print(f"transcribing raw mixture ({total_dur_s:.1f}s) with Whisper large-v3 ...")
    mix_model = whisper.load_model("large-v3", device=str(device))
    mix_result = mix_model.transcribe(
        input_mix.astype(np.float32),
        language="pl",
        initial_prompt="Rozmowa po polsku.",
        word_timestamps=True,
        temperature=0.0,
        condition_on_previous_text=False,
        verbose=False,
        fp16=(device.type == "cuda"),
    )
    mix_segments = [
        {"start": float(s["start"]), "end": float(s["end"]), "text": s["text"]}
        for s in mix_result["segments"]
        if s["text"].strip()
    ]
    with open(MIX_CACHE, "w", encoding="utf-8") as f:
        json.dump({"segments": mix_segments}, f, ensure_ascii=False, indent=2)
    print(f"cached -> {MIX_CACHE}")
    mix_model = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Build SegLST inputs: hyp = single-stream mixture; ref = same multi-speaker
# GT used in the cpWER cell above.
mix_hyp_seglst = SegLST([
    {
        "session_id": rid,
        "speaker": "mixture",
        "start_time": s["start"],
        "end_time": s["end"],
        "words": _normalize_text(s["text"]),
    }
    for s in mix_segments
])

ref_seglst = SegLST([
    {
        "session_id": rid,
        "speaker": spk,
        "start_time": float(u.start),
        "end_time": float(u.end),
        "words": _normalize_text(u.text),
    }
    for spk, utts in ref_utts.items()
    for u in utts
    if u.text.strip()
])

orc = orcwer(ref_seglst, mix_hyp_seglst)[rid]
mix_total_words = sum(len(_normalize_text(s["text"]).split()) for s in mix_segments)

print()
print(f"Mixture hyp: {len(mix_segments)} segments, {mix_total_words} words after normalisation")
print()
print(f"Mixture-Whisper ORC-WER = {orc.error_rate*100:5.2f}%   ({orc.errors}/{orc.length} words)")
print(f"Pipeline cpWER          = {result['cpwer']*100:5.2f}%   ({result['cp_errors']}/{result['cp_length']} words)")
print(f"Pipeline tcpWER         = {result['tcpwer']*100:5.2f}%   ({result['tcp_errors']}/{result['tcp_length']} words)")
delta_cp  = (result['cpwer']  - orc.error_rate) * 100
delta_tcp = (result['tcpwer'] - orc.error_rate) * 100
print(f"Δ cpWER  vs baseline    = {delta_cp:+.2f}pp   ({'pipeline helps' if delta_cp < 0 else 'pipeline hurts'})")
print(f"Δ tcpWER vs baseline    = {delta_tcp:+.2f}pp   ({'pipeline helps' if delta_tcp < 0 else 'pipeline hurts'})")

## Summary

All three layers in one block. Copy-paste into iteration notes when comparing pipeline configurations.

In [ ]:
summary = {
    "recording_id": rid,
    "duration_s": round(total_dur_s, 2),
    "gt_source": {"L": gt_L_src, "R": gt_R_src},
    "permutation": {
        "pipeline_s1_to_oracle": spk_for_estimate[0],
        "pipeline_s2_to_oracle": spk_for_estimate[1],
    },
    "layer1_der": {
        "no_collar":    {k: round(v, 4) for k, v in der_no_collar.items()    if isinstance(v, (int, float))},
        "collar_500ms": {k: round(v, 4) for k, v in der_collar_500.items() if isinstance(v, (int, float))},
    },
    "layer2_intrusive": {
        "L": {k: round(v, 3) for k, v in intr_L.items()},
        "R": {k: round(v, 3) for k, v in intr_R.items()},
    },
    "layer2_squim_non_intrusive": {
        "mixture":     {k: round(v, 3) for k, v in squim_mix.items()},
        "pipeline_s1": {k: round(v, 3) for k, v in squim_s1.items()},
        "pipeline_s2": {k: round(v, 3) for k, v in squim_s2.items()},
    },
    "layer3_wer": {
        "cpwer":           round(result["cpwer"],  4),
        "tcpwer":          round(result["tcpwer"], 4),
        "cp_assignment":   list(result["cp_assignment"]),
        "tcp_assignment":  list(result["tcp_assignment"]),
        "tcp_collar_s":    TCP_COLLAR_S,
    },
    "layer3b_mixture_baseline": {
        "orcwer":      round(float(orc.error_rate), 4),
        "errors":      int(orc.errors),
        "ref_words":   int(orc.length),
        "delta_cp_pp":  round(delta_cp, 2),
        "delta_tcp_pp": round(delta_tcp, 2),
    },
}
print(json.dumps(summary, indent=2, ensure_ascii=False))